In [ ]:
# --- Imports ---
import MDAnalysis as mda
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.colors import LinearSegmentedColormap

# === Define system ===
system_name = "name"

PRMTOP = "top_file.prmtop"
TRAJ = "traj_file.dcd"
DAT = "PCA_Contribution.dat"

# === Parameters ===
n_top = 10
os.makedirs("plots", exist_ok=True)

# === Define colormap ===
colors = [(0, 0, 1), (1, 1, 1), (1, 0, 0)]
cmap = LinearSegmentedColormap.from_list("blue_white_red", colors, N=256)

# === Function to compute DCCM ===
def compute_dccm(sel):
    positions = np.array([sel.positions for ts in sel.universe.trajectory])
    mean_positions = np.mean(positions, axis=0)
    fluctuations = positions - mean_positions
    n_frames, n_atoms, _ = fluctuations.shape

    dccm = np.zeros((n_atoms, n_atoms))
    for i in range(n_atoms):
        for j in range(n_atoms):
            num = np.sum(np.sum(fluctuations[:, i, :] * fluctuations[:, j, :], axis=1))
            den = np.sqrt(np.sum(np.sum(fluctuations[:, i, :]**2, axis=1)) *
                          np.sum(np.sum(fluctuations[:, j, :]**2, axis=1)))
            dccm[i, j] = num / den if den != 0 else 0
    return dccm

print(f"\n Processing system: {system_name}")

# Load top residues
df = pd.read_csv(DAT, delim_whitespace=True)
df_top = df.sort_values(by="Combined_Contribution(%)", ascending=False).head(n_top)
df_top = df_top.sort_values(by="ResSeq", ascending=True)

resids = df_top["ResSeq"].astype(int).tolist()
resnames = df_top["Residue"].tolist()

# Load MDAnalysis universe
u = mda.Universe(PRMTOP, TRAJ)
sel = u.select_atoms(f"name CA and resid {' '.join(map(str, resids))}")

print(f"Selected {len(sel)} CA atoms for DCCM computation.")

dccm = compute_dccm(sel)

# === Plot only lower triangle with manual borders ===
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(dccm, dtype=bool), k=1)

ax = sns.heatmap(
    dccm,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    mask=mask,
    xticklabels=resnames,
    yticklabels=resnames,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0,
    cbar_kws={'label': 'Correlation Coefficient (-1 to +1)'}
)

# Draw borders only for lower triangle (including diagonal)
for i in range(dccm.shape[0]):
    for j in range(dccm.shape[1]):
        if i >= j:
            ax.add_patch(
                plt.Rectangle((j, i), 1, 1,
                              fill=False,
                              edgecolor='black',
                              lw=1)
            )

plt.title(f"{system_name} — DCCM (Lower Triangle Only)",
          fontsize=14,
          fontweight='bold')
plt.xlabel("Residues", fontsize=12)
plt.ylabel("Residues", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()

outfile = f"plots/{system_name}_DCCM_lower_triangle_only.png"
plt.savefig(outfile, dpi=300)
plt.show()

print(f" Saved: {outfile}")